<a href="https://colab.research.google.com/github/chandupradeep18/Python_pandas_projects/blob/main/Google_Shopping_Feed_Health_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from itertools import combinations

# Set visual style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Cell 2: Generate Synthetic Product Feed Data
def generate_product_feed(n_products=100000, seed=42):
    """Generate a synthetic Google Shopping product feed with realistic correlated quality issues."""
    np.random.seed(seed)

    categories = [
        "Electronics > Smartphones",
        "Electronics > Laptops",
        "Apparel > Men's Clothing",
        "Apparel > Women's Clothing",
        "Home & Garden > Furniture",
        "Home & Garden > Kitchen",
        "Sports > Fitness Equipment",
        "Beauty > Skincare",
        "Toys > Board Games",
        "Food > Snacks"
    ]

    brands = [
        "TechCorp", "StyleMax", "HomeBase", "FitGear", "GlowUp",
        "SmartLife", "UrbanWear", "KitchenPro", "PlayWorld", "NatureBite",
        "EliteGadget", "ComfortZone", "ActiveEdge", "PureSkin", "FunFactory"
    ]

    conditions = ["new", "refurbished", "used"]
    availability_options = ["in_stock", "out_of_stock", "preorder"]
    seller_tiers = ["gold", "silver", "bronze"]

    ids = [f"PROD-{i:06d}" for i in range(n_products)]
    chosen_categories = np.random.choice(categories, n_products)
    chosen_brands = np.random.choice(brands, n_products)
    prices = np.round(np.random.lognormal(mean=3.5, sigma=1.0, size=n_products), 2)

    seller_tier = np.random.choice(seller_tiers, n_products, p=[0.3, 0.45, 0.25])

    tier_multipliers = {"gold": 0.3, "silver": 1.0, "bronze": 2.5}

    base_titles = []
    for i in range(n_products):
        brand = chosen_brands[i]
        cat = chosen_categories[i].split(" > ")[1]
        adjectives = ["Premium", "Classic", "Pro", "Ultra", "Essential", "Deluxe"]
        adj = np.random.choice(adjectives)
        title = f"{brand} {adj} {cat} - Model {np.random.randint(100, 999)}"
        base_titles.append(title)

    gtins = []
    for i in range(n_products):
        base_prob = 0.15
        prob = min(base_prob * tier_multipliers[seller_tier[i]], 0.95)
        roll = np.random.random()
        if roll < prob:
            gtins.append(None)
        elif roll < prob + 0.03:
            gtins.append(str(np.random.randint(100, 999)))
        else:
            gtins.append(str(np.random.randint(1000000000000, 9999999999999)))

    df = pd.DataFrame({
        "id": ids,
        "title": base_titles,
        "description": [
            f"High-quality {chosen_categories[i].split(' > ')[1].lower()} from {chosen_brands[i]}. "
            f"Perfect for everyday use." if np.random.random() > 0.06 * tier_multipliers[seller_tier[i]]
            else None
            for i in range(n_products)
        ],
        "link": [f"https://example.com/products/{ids[i].lower()}" for i in range(n_products)],
        "image_link": [
            f"https://example.com/images/{ids[i].lower()}.jpg"
            if np.random.random() > 0.05 * tier_multipliers[seller_tier[i]]
            else None
            for i in range(n_products)
        ],
        "availability": np.random.choice(availability_options, n_products, p=[0.75, 0.20, 0.05]),
        "price": prices,
        "brand": [
            chosen_brands[i] if np.random.random() > 0.08 * tier_multipliers[seller_tier[i]] else None
            for i in range(n_products)
        ],
        "gtin": gtins,
        "google_product_category": chosen_categories,
        "condition": np.random.choice(conditions, n_products, p=[0.85, 0.10, 0.05]),
        "seller_tier": seller_tier
    })

    for i in range(n_products):
        if np.random.random() < 0.03 * tier_multipliers[seller_tier[i]]:
            df.at[i, "price"] = np.random.choice([0, -1, -5.99])

    for i in range(n_products):
        prob = 0.10 * tier_multipliers[seller_tier[i]]
        if np.random.random() < prob:
            issue_type = np.random.choice(["short", "caps", "promo"])
            if issue_type == "short":
                df.at[i, "title"] = "Item"
            elif issue_type == "caps":
                df.at[i, "title"] = df.at[i, "title"].upper()
            else:
                df.at[i, "title"] = "FREE SHIPPING! " + df.at[i, "title"] + " BEST DEAL!!!"

    return df

# Generate the dataset
products = generate_product_feed()
products.to_csv('product.csv')
products.to_json('product.json',indent=2)

In [ ]:
print(f"Dataset shape: {products.shape}")
print(products.columns.to_list())
print(products.dtypes)

Dataset shape: (100000, 12)
['id', 'title', 'description', 'link', 'image_link', 'availability', 'price', 'brand', 'gtin', 'google_product_category', 'condition', 'seller_tier']
id                          object
title                       object
description                 object
link                        object
image_link                  object
availability                object
price                      float64
brand                       object
gtin                        object
google_product_category     object
condition                   object
seller_tier                 object
dtype: object


In [ ]:
print(products.isnull().sum().sort_values(ascending=False))
for tier in ["gold", "silver", "bronze"]:
  tier_df=products[products['seller_tier']==tier]
  null_rate =products.isnull().sum().sum() / (len(tier_df)*len(products.columns)) *100
  #print(f"Tier : {tier} \n{null_rate}")
  print(f"{tier.upper():8s} | {len(tier_df):,} products | Null rate: {null_rate:.1f}%")

gtin                       17567
brand                       9261
description                 6885
image_link                  5844
link                           0
title                          0
id                             0
availability                   0
price                          0
google_product_category        0
condition                      0
seller_tier                    0
dtype: int64
GOLD     | 30,084 products | Null rate: 11.0%
SILVER   | 45,008 products | Null rate: 7.3%
BRONZE   | 24,908 products | Null rate: 13.2%
